### Pinecone을 이용한 벡터 DB 구축하기

#### ChromaDB의 단점
- 배포시 서버가 너무 자주 다운됨, 그러면서 메모리에 있던 크로마DB가 사라짐
- 대안: 클라우드DB 사용(Azure AI Search, Back-to-DB) => 문제: 백터DB를 직접 작성(해당 패키지를 이용해 코드를 다시 만들어야함)
-  LangChain에서는 자체적으로 관리되는 DB가 많기 때문에 별도의 코드를 작성하지 않고  Database 변경하면 됨

#### Pinecone을 활용한 DB 구축
- Pinecone은 고성능 벡터 데이터베이스로, AI 및 머신러닝 애플리케이션을 위한 효율적인 벡터 저장 및 검색 솔루션
- https://wikidocs.net/252407 참조
- 파이콘 랭체인 : https://python.langchain.com/v0.2/docs/integrations/vectorstores/pinecone/
- 무료로 5개까지 사용 가능(테스트용으로 사용)

In [ ]:
%pip install -qU pinecone
%pip install -qU langchain langchain-core langchain-community langchain-openai
%pip install -qU docx2txt pypdf
%pip install -qU langchain-text-splitters langchain-pinecone

- 문서를 읽어와 분활하기

In [ ]:
import docx2txt
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

file_path ="./docs/소득세법_20260701.docx"

text = docx2txt.process(file_path)

# Document 생성
document = Document(
    page_content=text, 
    metadata={"source": file_path})

# print(document.page_content[:500])

# 텍스트 분할
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents([document])


총 313개의 청크로 분할되었습니다.


In [ ]:
# 결과 확인
print(f"총 {len(chunks)}개의 청크로 분할되었습니다.")
print(f"첫 번째 청크 내용:\n{chunks[0].page_content}")

### Pinecone DB 사용
1. 회원가입 후 API Key 생성
2. index 생성
3. 코드 적용

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader, PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone
from dotenv import load_dotenv
import os

In [4]:
# 1. 환경 변수 로드 및 문서 준비(docx)

load_dotenv()

loader = Docx2txtLoader("./docs/소득세법_20260701.docx")

In [9]:
# 2. 문서 로드 및 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=200,
    separators=['제', '\n\n', '\n', ' ', '']
)

# documents = loader.load()
# chunks = text_splitter.split_documents(documents)

chunks = loader.load_and_split(text_splitter=text_splitter)

In [10]:
print(f"총 {len(chunks)}개의 청크로 분할되었습니다.")

총 311개의 청크로 분할되었습니다.


In [11]:
# 3. 문서 임베딩 및 벡터 스토어 생성
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

database = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="my-tax-index"
)

In [18]:
# 4.Recursive 생성
recursive = database.as_retriever(
    search_type="similarity",  # 유사도 기반 검색
    search_kwargs={"k": 3}     # 검색할 유사 문서 개수
    )

In [ ]:
# 5. Prompt 생성 => 객체 형태로 생성
from langchain_core.prompts import ChatPromptTemplate

prompt_template = ChatPromptTemplate.from_template(
    '''
    [Identity]
    당신은 한국의 소득세법 전문가입니다.
    [Context] 제공된 내용만 이용해서 사용자의 질문에 친절하게 답변해 주세요.
    [Context]에 관련 내용이 없다면 "제공된 소득세법 문서에는 관련 내용이 없습니다."라고 답변해 주세요.
    마지막에는 반드시 [출처]를 명시해 주세요.

    [Context]
    {context}

    [Question]
    {query}
    '''
)

In [ ]:
# pip install -qU langchain-google-genai

In [32]:
# 6. LLM 생성
from langchain.chat_models import init_chat_model

llm = init_chat_model(
    model="gpt-4o", 
    model_provider='openai',    # "google-genai", "ollama"
    temperature=0
)

In [30]:
llm.invoke('안녕하세요.').content

'안녕하세요! 어떻게 도와드릴까요?'

In [33]:
# 7. 문자열 출력 형식
def format_response(docs):
    return "\n\n".join([f"출처: {doc.metadata['source']}\n내용: {doc.page_content}" for doc in docs])

In [34]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


rag_chain = (
    {"context": recursive | format_response,  # recursive 결과를 format_response 함수에 전달
     "query": RunnablePassthrough()}          # 전달한 query를 그대로 사용
    | prompt_template
    | llm
    | StrOutputParser()
)

In [45]:
query = '종합과세 표준에 대해 설명해주세요'
result = rag_chain.invoke(query)

print(result)

종합소득과세표준은 개인의 다양한 소득을 합산하여 과세하는 기준 금액을 의미합니다. 이는 여러 종류의 소득을 합산한 후, 법에서 정한 공제를 적용하여 계산됩니다. 구체적으로, 종합소득과세표준은 다음과 같은 절차로 계산됩니다:

1. **소득의 합산**: 이자소득, 배당소득, 사업소득, 근로소득, 연금소득, 기타소득 등 다양한 소득 항목을 합산하여 종합소득금액을 계산합니다. 이 과정에서 특정 소득은 합산에서 제외될 수 있습니다. 예를 들어, 조세특례제한법에 따라 과세되지 않는 소득이나 일용근로자의 근로소득 등은 합산되지 않습니다.

2. **공제 적용**: 종합소득금액에서 법에서 정한 공제를 적용합니다. 여기에는 기본공제, 추가공제, 특별공제 등이 포함될 수 있습니다.

3. **과세표준 계산**: 공제를 적용한 후의 금액이 종합소득과세표준이 됩니다. 이 금액을 기준으로 소득세가 부과됩니다.

종합소득과세표준은 개인의 소득세 부담을 결정하는 중요한 요소로, 다양한 소득을 종합적으로 고려하여 공정한 과세를 목표로 합니다.
